# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is accessible via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields, referencing all entities by their `@id`s as required by the Croissant schema.

We'll list all available record sets and, for each, show its fields and columns with their `@id`s.

In [ ]:
# List available record sets and their fields by @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for f in fields:
        if isinstance(f, dict):
            print(f"    Field @id: {f['@id']}  (name: {f.get('name')})")
            columns = f.get('column', [])
            if isinstance(columns, dict):
                columns = [columns]
            for c in columns:
                if isinstance(c, dict):
                    print(f"      Column @id: {c['@id']}  (name: {c.get('name')})")
    print()

## 3. Data Extraction
We'll load data from each record set into a pandas DataFrame for analysis.

Below, we select record set and field/column `@id`s based on the previous overview.

*All references use `@id` as required.*

In [ ]:
# Use the discovered record sets' @id values to extract records
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} records.")
        print(f"  Columns: {list(df.columns)}\n")
    except Exception as e:
        print(f"  Could not load records for {record_set_id}: {e}\n")

# If there are any dataframes loaded, display the head of the largest
if dataframes:
    largest_rs = max(dataframes, key=lambda k: len(dataframes[k]))
    print(f"Displaying the first 5 records for RecordSet @id: {largest_rs}")
    display(dataframes[largest_rs].head())
else:
    print("No record sets loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common exploratory data steps, such as filtering records based on a numeric field, normalizing numeric fields, and grouping by another field.

In this example, we'll pick a numeric field from the dataset by its `@id`. You may need to adjust the field `@id` and record set depending on your overview results.

In [ ]:
# --- EDA: Filtering and Normalizing ---
# You must adjust these variables based on the previous data overview:
example_record_set_id = None  # Replace with an actual record set @id
numeric_field_id = None       # Replace with an actual numeric field @id (column name)
group_field_id = None         # Replace with actual field @id to group by

# Try to automatically select one numeric field if possible
import numpy as np
for rs_id, df in dataframes.items():
    num_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if num_fields:
        example_record_set_id = rs_id
        numeric_field_id = num_fields[0]
        print(f"Chose record set {rs_id} and numeric field '{numeric_field_id}' for EDA.")
        # For the group field, pick first non-numeric column
        non_num = [col for col in df.columns if df[col].dtype == object]
        if non_num:
            group_field_id = non_num[0]
        break

if not example_record_set_id or not numeric_field_id:
    print("No suitable record set or numeric field found for EDA.")
else:
    df = dataframes[example_record_set_id]
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0

    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (from RecordSet {example_record_set_id}):")
    display(filtered_df.head())

    # Normalization
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by group_field_id if it exists:
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. You may need to update the field `@id`s depending on the dataset contents.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Visualize the distribution of the numeric field (from EDA)
if example_record_set_id and numeric_field_id:
    df = dataframes[example_record_set_id]
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in RecordSet {example_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # Scatter plot of numeric vs. group if available
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated loading and exploring a Croissant-formatted dataset using `mlcroissant`, referencing all data and columns by their `@id`. We performed data overview, extraction, basic EDA, and visualized some key features.

You can continue by running your own analyses or extending the notebook for other datasets with Croissant schemas.